### 과제 1. 할리스커피

- https://www.hollys.co.kr/store/korea/korStore2.do


#### 요구사항
1. 총 10페이지를 순회할 것 (페이지를 넘기며 URL이 바뀌는 것을 확인)
2. 추출 필드: 지역 / 매장명 / 현황 / 주소 / 매장 서비스 / 전화번호
3. 매장 서비스는 리스트로 담을 것 (아이콘이 여러 개인 매장이 있음)
4. 결과를 hollys.csv로 저장할 것

```py
[{'지역': '서울 동대문구',
  '매장명': '경희대 경영대점',
  '현황': '영업중',
  '주소': '서울특별시 동대문구 경희대로 26 (회기동) 경영대학 3층',
  '매장 서비스': ['주차'],
  '전화번호': '.'},
 ...]
```

In [1]:
import requests
from bs4 import BeautifulSoup

In [4]:
URL = 'https://www.hollys.co.kr/store/korea/korStore2.do'

params = {
    'sido': '',
    'gugun': '',
    'store': '',
    'pageNo': 1,
}

res = requests.get(URL, params=params)
# res.status_code
res.raise_for_status()

In [ ]:
res.text.find('대전도안마을점') # 정적 웹사이트

49392

In [5]:
soup = BeautifulSoup(res.text, 'html.parser')

In [7]:
items = soup.select('.tb_store tbody tr')

def get_text(tag):
    return tag.text.strip() if tag else ''

stores = []
for item in items:
    # 지역, 매장명 등등 찾아서 정리
    tds = item.select('td')

    # 매장서비스 정리 (img 태그들로 이루어짐)
    imgs = tds[4].select('img')
    services = []
    for img in imgs:
        services.append(img.attrs['alt'])
    # services = [img.attrs['alt'] for img in imgs]

    stores.append({
        '지역': get_text(tds[0]),
        '매장명': get_text(tds[1]),
        '현황': get_text(tds[2]),
        '주소': get_text(tds[3]),
        '매장서비스': services,
        '전화번호': get_text(tds[5]),
    })

stores

[{'지역': '대전 유성구',
  '매장명': '대전도안마을점',
  '현황': '영업중',
  '주소': '대전광역시 유성구 도안대로 560 (도안마을1단지) /봉명동 1024',
  '매장서비스': [],
  '전화번호': '042-826-6080'},
 {'지역': '충북 음성군',
  '매장명': '국립소방병원점',
  '현황': '영업중',
  '주소': '충청북도 음성군 맹동면 용두4길 19 (국립소방병원) /두성리 1531',
  '매장서비스': ['주차'],
  '전화번호': '042-882-0240'},
 {'지역': '경기 용인시 수지구',
  '매장명': '용인수지구청점',
  '현황': '영업중',
  '주소': '경기도 용인시 수지구 풍덕천로 119 (수지로얄스포츠센타), 109호 .',
  '매장서비스': ['주차'],
  '전화번호': '031-266-3011'},
 {'지역': '서울 동작구',
  '매장명': '이수점',
  '현황': '영업중',
  '주소': '서울특별시 동작구 동작대로25길 16 (사당동) 1층~2층',
  '매장서비스': [],
  '전화번호': '02-3478-9029'},
 {'지역': '충남 아산시',
  '매장명': '아산삼성스토어점',
  '현황': '영업중',
  '주소': '충청남도 아산시 모종로 9-17, 삼성스토어 1층 .',
  '매장서비스': ['주차'],
  '전화번호': '041-543-1233'},
 {'지역': '부산 해운대구',
  '매장명': '부산해운대엘시티점',
  '현황': '영업중',
  '주소': '부산 해운대구 달맞이길 30,포디움동  2층 2042~2043호',
  '매장서비스': ['테라스', '주차'],
  '전화번호': '051-746-3547'},
 {'지역': '경기 용인시 기흥구',
  '매장명': '용인동백점',
  '현황': '영업중',
  '주소': '경기도 용인시 기흥구 동백중앙로 283 (골드프라자 D동),  203호,204호',
  '매장서비

In [ ]:
temp = {
    'a': 1,
    'b': 3
}

# 딕셔너리 언패킹
{**temp, 'pageNo': 3}

# pymysql.connect(**db_config)
# pymysql.connect(host='localhost', port=3306)

{'a': 1, 'b': 3, 'pageNo': 3}

In [13]:
# 할리스 최종 코드
import time

URL = 'https://www.hollys.co.kr/store/korea/korStore2.do'

PARAMS = {
    'sido': '',
    'gugun': '',
    'store': '',
}

def get_text(tag):
    return tag.text.strip() if tag else ''

def fetch(page: int) -> str:
    res = requests.get(URL, params={**PARAMS, 'pageNo': page})
    # res.status_code
    res.raise_for_status()
    return res.text

def parse(html: str) -> list[dict]:
    soup = BeautifulSoup(html, 'html.parser')
    items = soup.select('.tb_store tbody tr')
    stores = []
    for item in items:
        # 지역, 매장명 등등 찾아서 정리
        tds = item.select('td')

        # 매장서비스 정리 (img 태그들로 이루어짐)
        imgs = tds[4].select('img')
        services = []
        for img in imgs:
            services.append(img.attrs['alt'])
        # services = [img.attrs['alt'] for img in imgs]

        stores.append({
            '지역': get_text(tds[0]),
            '매장명': get_text(tds[1]),
            '현황': get_text(tds[2]),
            '주소': get_text(tds[3]),
            '매장서비스': services,
            '전화번호': get_text(tds[5]),
        })
    return stores

# main logic

stores = []
for page in range(1, 11):
    html = fetch(page)
    result = parse(html)
    stores.extend(result)

    time.sleep(0.7) # 예의상 조금 기다렸다가 다음페이지 수집

TypeError: sequence item 4: expected str instance, list found

In [30]:
# stores

import pandas as pd
def save(datas, filename):
    pd.DataFrame(datas).to_csv(filename, index=None)
    
# with open('./hollys.csv', 'w') as f:
#     f.write(','.join(stores[0].keys()))
#     f.write('\n')
#     for store in stores:
#         f.write(','.join(map(str, store.values())) + '\n')


### 과제 2. 알라딘 베스트셀러 수집

- https://www.aladin.co.kr/shop/common/wbest.aspx?BranchType=1

#### 요구사항

- 추출 필드: 카테고리 / 제목 / 저자 / 할인가격 / 이미지 URL
- 이미지 URL은 <img> 태그의 속성에서 가져올 것
- 결과를 aladin_bestseller.csv로 저장할 것
- 추가학습: 총 500위까지 수집해주세요.

In [1]:
import requests
from bs4 import BeautifulSoup

In [ ]:
URL = 'https://www.aladin.co.kr/shop/common/wbest.aspx'

PARAMS = {
    'BestType': 'Bestseller',
    'BranchType': 1,
    'CID': 0,
    'cnt': 1000,
    'SortOrder': 1
}

res = requests.get(URL, params={**PARAMS, 'page': 1})

res.raise_for_status()

In [3]:
res.text.find('세네카')

144473

In [4]:
soup = BeautifulSoup(res.text, 'html.parser')

In [6]:
def get_text(tag):
    return tag.text.strip() if tag else ''

In [ ]:
# 박스 찾기

items = soup.select('.ss_book_box')

books = []
for item in items:
    # 저자 정리
    lis = item.select('.ss_book_list:nth-child(1) li')
    if len(lis) == 5:
        author = lis[2].select_one('a')
    else:
        author = lis[1].select_one('a')

    # class가 cover_area로 시작하는 img 태그 중 뒤에 것
    img = item.select('div[class^="cover_area"] img')[-1]

    books.append({
        '카테고리': get_text(item.select_one('.tit_category')),
        '제목': get_text(item.select_one('.bo3')),
        '저자': get_text(author),
        '할인가격': get_text(item.select_one('.ss_p2')),
        '이미지': img.attrs['src'],
    })

books

[{'카테고리': '[국내도서]',
  '제목': '세네카, 오늘을 빼앗기고 있는 당신에게',
  '저자': '루키우스 안나이우스 세네카',
  '할인가격': '16,200원',
  '이미지': 'https://image.aladin.co.kr/product/39640/49/cover200/k872130175_1.jpg'},
 {'카테고리': '[국내도서]',
  '제목': '그랬다고 적었다',
  '저자': '김애란',
  '할인가격': '15,300원',
  '이미지': 'https://image.aladin.co.kr/product/40019/36/cover200/k742130236_1.jpg'},
 {'카테고리': '[국내도서]',
  '제목': '싯다르타',
  '저자': '헤르만 헤세',
  '할인가격': '7,200원',
  '이미지': 'https://image.aladin.co.kr/product/32/95/cover200/s062934786_1.jpg'},
 {'카테고리': '[국내도서]',
  '제목': '오뒷세이아',
  '저자': '호메로스',
  '할인가격': '22,500원',
  '이미지': 'https://image.aladin.co.kr/product/39940/12/cover200/8932476462_2.jpg'},
 {'카테고리': '[국내도서]',
  '제목': '오디세이아 (국내 유일 명화 104점 수록 완역본)',
  '저자': '호메로스',
  '할인가격': '24,300원',
  '이미지': 'https://image.aladin.co.kr/product/36239/0/cover200/k062038716_2.jpg'},
 {'카테고리': '[국내도서]',
  '제목': '찌니주의보',
  '저자': '정지아',
  '할인가격': '15,120원',
  '이미지': 'https://image.aladin.co.kr/product/40013/75/cover200/8936439995_1.jpg'},
 {'카테고리': '[

In [27]:
def fetch(page):
    res = requests.get(URL, params={**PARAMS, 'page': page})
    res.raise_for_status()
    return res.text

def parse(html):
    soup = BeautifulSoup(html, 'html.parser')
    items = soup.select('.ss_book_box')

    books = []
    for item in items:
        # 저자 정리
        lis = item.select('.ss_book_list:nth-child(1) li')
        if len(lis) == 5:
            author = lis[2].select_one('a')
        else:
            author = lis[1].select_one('a')

        # class가 cover_area로 시작하는 img 태그 중 뒤에 것
        img = item.select('div[class^="cover_area"] img')[-1]

        books.append({
            '카테고리': get_text(item.select_one('.tit_category')),
            '제목': get_text(item.select_one('.bo3')),
            '저자': get_text(author),
            '할인가격': get_text(item.select_one('.ss_p2')),
            '이미지': img.attrs['src'],
        })
    return books

books = []
for page in range(1, 11):
    html = fetch(page)
    result = parse(html)
    books.extend(result)

books

[{'카테고리': '[국내도서]',
  '제목': '세네카, 오늘을 빼앗기고 있는 당신에게',
  '저자': '루키우스 안나이우스 세네카',
  '할인가격': '16,200원',
  '이미지': 'https://image.aladin.co.kr/product/39640/49/cover200/k872130175_1.jpg'},
 {'카테고리': '[국내도서]',
  '제목': '그랬다고 적었다',
  '저자': '김애란',
  '할인가격': '15,300원',
  '이미지': 'https://image.aladin.co.kr/product/40019/36/cover200/k742130236_1.jpg'},
 {'카테고리': '[국내도서]',
  '제목': '싯다르타',
  '저자': '헤르만 헤세',
  '할인가격': '7,200원',
  '이미지': 'https://image.aladin.co.kr/product/32/95/cover200/s062934786_1.jpg'},
 {'카테고리': '[국내도서]',
  '제목': '오뒷세이아',
  '저자': '호메로스',
  '할인가격': '22,500원',
  '이미지': 'https://image.aladin.co.kr/product/39940/12/cover200/8932476462_2.jpg'},
 {'카테고리': '[국내도서]',
  '제목': '오디세이아 (국내 유일 명화 104점 수록 완역본)',
  '저자': '호메로스',
  '할인가격': '24,300원',
  '이미지': 'https://image.aladin.co.kr/product/36239/0/cover200/k062038716_2.jpg'},
 {'카테고리': '[국내도서]',
  '제목': '찌니주의보',
  '저자': '정지아',
  '할인가격': '15,120원',
  '이미지': 'https://image.aladin.co.kr/product/40013/75/cover200/8936439995_1.jpg'},
 {'카테고리': '[

In [31]:
len(books)

save(books, 'aladin_bestseller.csv')